In [14]:
#the function that safes the data and creates a new text file with the data in it

from pathlib import Path
from datetime import datetime
import numpy as np

def create_minimization_filename(output_dir):
    """
    Erzeugt einmalig einen Dateinamen mit Datum und Uhrzeit.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    filename = output_dir / f"minimization_data_{timestamp}.txt"

    return filename


def write_minimization_step_to_txt(
        filename,
        step,
        positions,
        epot,
        forces,
        append=True
    ):
    """
    Schreibt die Daten eines Minimierungsschrittes in eine bestehende TXT-Datei.
    """

    filename = Path(filename)

    positions = np.asarray(positions)
    forces = np.asarray(forces)

    if positions.shape != forces.shape:
        raise ValueError("positions und forces müssen dieselbe Form haben, z.B. (n_particles, 3).")

    if positions.shape[1] != 3:
        raise ValueError("positions und forces müssen jeweils drei Spalten haben.")

    n_particles = positions.shape[0]

    step_column = np.full((n_particles, 1), step)
    particle_column = np.arange(n_particles).reshape(-1, 1)
    epot_column = np.full((n_particles, 1), epot)

    data = np.column_stack([
        step_column,
        particle_column,
        epot_column,
        positions,
        forces
    ])

    header = "step particle E_pot x_nm y_nm z_nm Fx Fy Fz"

    mode = "a" if append else "w"
    write_header = (not filename.exists()) or (not append)

    with open(filename, mode) as f:
        if write_header:
            f.write(header + "\n")

        np.savetxt(
            f,
            data,
            fmt=[
                "%d", "%d", "%.12e",
                "%.12e", "%.12e", "%.12e",
                "%.12e", "%.12e", "%.12e"
            ]
        )

In [ ]:
#visualize the function

In [12]:
positions_list = [
    [[1.00, 2.00, 3.00], [2.00, 3.00, 4.00], [3.00, 4.00, 5.00]],
    [[1.15, 2.00, 3.00], [2.00, 3.15, 4.00], [3.00, 4.00, 4.85]],
    [[1.30, 2.00, 3.00], [2.00, 3.30, 4.00], [3.00, 4.00, 4.70]],
    [[1.45, 2.00, 3.00], [2.00, 3.45, 4.00], [3.00, 4.00, 4.55]],
    [[1.60, 2.00, 3.00], [2.00, 3.60, 4.00], [3.00, 4.00, 4.40]],
    [[1.75, 2.00, 3.00], [2.00, 3.75, 4.00], [3.00, 4.00, 4.25]],
    [[1.90, 2.00, 3.00], [2.00, 3.90, 4.00], [3.00, 4.00, 4.10]],
    [[2.05, 2.00, 3.00], [2.00, 4.05, 4.00], [3.00, 4.00, 3.95]],
    [[2.20, 2.00, 3.00], [2.00, 4.20, 4.00], [3.00, 4.00, 3.80]],
    [[2.35, 2.00, 3.00], [2.00, 4.35, 4.00], [3.00, 4.00, 3.65]],
]


epot_list = [
    -10.0,
    -10.5,
    -11.0,
    -11.4,
    -11.7,
    -11.9,
    -12.05,
    -12.15,
    -12.21,
    -12.25,
]


forces_list = [
    [[ 0.10, -0.05,  0.02], [-0.08,  0.03, -0.01], [ 0.06,  0.02, -0.04]],
    [[ 0.09, -0.045, 0.018], [-0.07,  0.028, -0.009], [ 0.055, 0.018, -0.035]],
    [[ 0.08, -0.040, 0.016], [-0.06,  0.026, -0.008], [ 0.050, 0.016, -0.030]],
    [[ 0.07, -0.035, 0.014], [-0.05,  0.024, -0.007], [ 0.045, 0.014, -0.025]],
    [[ 0.06, -0.030, 0.012], [-0.04,  0.022, -0.006], [ 0.040, 0.012, -0.020]],
    [[ 0.05, -0.025, 0.010], [-0.035, 0.020, -0.005], [ 0.035, 0.010, -0.017]],
    [[ 0.04, -0.020, 0.008], [-0.030, 0.018, -0.004], [ 0.030, 0.008, -0.014]],
    [[ 0.03, -0.015, 0.006], [-0.025, 0.016, -0.003], [ 0.025, 0.006, -0.011]],
    [[ 0.02, -0.010, 0.004], [-0.020, 0.014, -0.002], [ 0.020, 0.004, -0.008]],
    [[ 0.01, -0.005, 0.002], [-0.015, 0.012, -0.001], [ 0.015, 0.002, -0.005]],
]



In [15]:
name = create_minimization_filename("minimization_output") 

for i in range(len(positions_list)):
    write_minimization_step_to_txt(name, i, positions_list[i], epot_list[i], forces_list[i], append=True)

In [17]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go


def plot_minimization_points_with_epot(filename, box_length):
    """
    Erstellt einen 3D-Plot mit beweglichen Punktteilchen und E_pot-Anzeige.

    Inputs
    ------
    filename : str oder Path
        Pfad zur TXT-Datei.

    box_length : float
        Länge der kubischen Simulationsbox in nm.

    Erwartete Spalten in der TXT:
    step particle E_pot x_nm y_nm z_nm Fx Fy Fz
    """

    filename = Path(filename)

    # TXT-Datei einlesen
    df = pd.read_csv(filename, sep=r"\s+")

    # Alle Steps sortieren
    steps = sorted(df["step"].unique())

    # Achsengrenzen über die echte Boxlänge festlegen
    x_range = [0, box_length]
    y_range = [0, box_length]
    z_range = [0, box_length]

    def data_for_step(step):
        """
        Gibt die Punktteilchen für genau einen Step zurück.
        """
        d = df[df["step"] == step]

        epot = d["E_pot"].iloc[0]

        trace = go.Scatter3d(
            x=d["x_nm"],
            y=d["y_nm"],
            z=d["z_nm"],
            mode="markers",
            marker=dict(
                size=6
            ),
            text=[
                f"Step: {step}<br>"
                f"Teilchen: {particle}<br>"
                f"x = {x:.4f} nm<br>"
                f"y = {y:.4f} nm<br>"
                f"z = {z:.4f} nm<br>"
                f"E_pot = {epot:.6e}"
                for particle, x, y, z in zip(
                    d["particle"], d["x_nm"], d["y_nm"], d["z_nm"]
                )
            ],
            hoverinfo="text",
            name="Teilchen"
        )

        return trace, epot

    # Erster Step
    first_step = steps[0]
    first_trace, first_epot = data_for_step(first_step)

    # Figure mit erstem Step starten
    fig = go.Figure(data=[first_trace])

    # Frames für alle Steps erstellen
    frames = []

    for step in steps:
        trace, epot = data_for_step(step)

        frame = go.Frame(
            data=[trace],
            name=str(step),
            layout=go.Layout(
                annotations=[
                    dict(
                        text=f"Step: {step} | E_pot = {epot:.6e}",
                        x=0.5,
                        y=1.05,
                        xref="paper",
                        yref="paper",
                        showarrow=False,
                        font=dict(size=16)
                    )
                ]
            )
        )

        frames.append(frame)

    fig.frames = frames

    # Slider konfigurieren
    slider_steps = []

    for step in steps:
        slider_step = dict(
            method="animate",
            label=str(step),
            args=[
                [str(step)],
                dict(
                    mode="immediate",
                    frame=dict(duration=0, redraw=True),
                    transition=dict(duration=0)
                )
            ]
        )

        slider_steps.append(slider_step)

    fig.update_layout(
        title="Minimierung: Bewegung der Punktteilchen",
        scene=dict(
            xaxis=dict(title="x / nm", range=x_range),
            yaxis=dict(title="y / nm", range=y_range),
            zaxis=dict(title="z / nm", range=z_range),
            aspectmode="cube"
        ),
        annotations=[
            dict(
                text=f"Step: {first_step} | E_pot = {first_epot:.6e}",
                x=0.5,
                y=1.05,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=16)
            )
        ],
        sliders=[
            dict(
                active=0,
                currentvalue=dict(prefix="Step: "),
                steps=slider_steps
            )
        ]
    )

    fig.show()

In [20]:
plot_minimization_points_with_epot(
    "minimization_output/minimization_data_2026-07-08_20-17-01.txt", 6
)